# Cancer🔬 Classification: Baseline with ⚡`lightning`

**It is continuation of baseline: https://www.kaggle.com/code/jirkaborovec/cancer-subtype-baseline-with-lightning-timm**

### Difference to the baseline

The baseline was using only thumbnails and extracting limited details by random crop.
In this case we take random tile from a particular WSI and assume it is reprentative for whole tissue

**NOTE:** in several cases we can hit edge cases when no meanigful tissue is present

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch


DATASET_FOLDER = "/kaggle/input/UBC-OCEAN/"
DATASET_IMAGES = "/kaggle/input/tiles-of-cancer-2048px-scale-0-25/"
BATCH_SIZE = 2
BAG_SIZE = 6
INPUT_SIZE = 512
LR = 3e-5
ACCUMULATE_GRAD_BATCHES = 16
EPOCHS = 60 if torch.cuda.is_available() else 2

## Checkout some labels

In [ ]:
df_train = pd.read_csv(os.path.join(DATASET_FOLDER, "train.csv"))
# labels = list(df_train["label"].unique())
print(f"Dataset/train size: {len(df_train)}")
display(df_train.head())

In [ ]:
_= df_train[["label"]].value_counts().plot.pie(autopct='%1.1f%%', ylabel="label", figsize=(3,3))

### Show some samples 🖼️ per class

Note that not all images has thumbnails

In [ ]:
import matplotlib.pyplot as plt

nb_samples = 6
n, m = len(np.unique(df_train['label'])), nb_samples,
fig, axarr = plt.subplots(nrows=n, ncols=m, figsize=(m * 2, n * 2))
for ilb, (lb, df_) in enumerate(df_train.groupby('label')):
    img_ids = list(df_['image_id'])
    for i in range(m):
        if i == 0:
            axarr[ilb, i].set_title(f"{lb} #{len(df_)}")
        ls_imgs = glob.glob(os.path.join(DATASET_IMAGES, str(img_ids[i]), "*.png"))
        img_path = ls_imgs[0]
        img = plt.imread(img_path)
        mask = np.sum(img[..., :3], axis=2) == 0
        img[mask, :] = 255
        axarr[ilb, i].imshow(img)
        # axarr[ilb, i].set_xticks([])
        # axarr[ilb, i].set_yticks([])
_= plt.axis('off')

## Data pre-processing

### Color 🦩 normalizations

In [ ]:
img_color_mean = [0.8721593659261734, 0.7799686061900686, 0.8644588534918227]
img_color_std = [0.08258995918115268, 0.10991684444009092, 0.06839816226731532]


## Dataset & DataModule

Creating standard PyTorch dataset to define how the data shall be loaded and set representations. We define the sample pair as:
- RGB image
- one-hot lable encding

A DataModule standardizes the training, val, test splits, data preparation and transforms. The main advantage is consistent data splits, data preparation and transforms across models.

In [ ]:
!pip install -U -q pytorch-lightning

In [ ]:
import os
import torch
import random
from PIL import Image
from torch.utils.data import Dataset

class CancerTilesDataset(Dataset):
    split: float = 0.90

    def __init__(
        self,
        df_data,
        path_img_dir,
        bag_size,
        transforms = None,
        mode: str = 'train',
        labels_lut = None,
        white_thr: int = 225,
        thr_max_bg: float = 0.2,
    ):
        assert os.path.isdir(path_img_dir)
        self.path_img_dir = path_img_dir
        self.transforms = transforms
        self.mode = mode
        self.white_thr = white_thr
        self.thr_max_bg = thr_max_bg

        self.data = df_data
        self.labels_unique = sorted(self.data["label"].unique())
        self.labels_lut = labels_lut or {lb: i for i, lb in enumerate(self.labels_unique)}
        # shuffle data
        self.data = self.data.sample(frac=1, random_state=42).reset_index(drop=True)
        self.bag_size = bag_size

        # split dataset
        assert 0.0 <= self.split <= 1.0
        frac = int(self.split * len(self.data))
        self.data = self.data[:frac] if mode == 'train' else self.data[frac:]
        self.img_dirs = [glob.glob(os.path.join(path_img_dir, str(idx), "*.png")) for idx in self.data["image_id"]]
        #print(f"missing: {sum([not os.path.isfile(os.path.join(self.path_img_dir, im))
        #                       for im in self.img_names])}")
        self.labels = list(self.data['label'])

    @property
    def num_classes(self) -> int:
        return len(self.labels_lut)

    def to_one_hot(self, label: str) -> tuple:
        one_hot = [0] * self.num_classes
        one_hot[self.labels_lut[label]] = 1
        return tuple(one_hot)
    
    def pad_bag(self, bag):
        c_size = len(bag)
        p_size = self.bag_size-c_size
        empty_tile = np.zeros_like(bag[0])
        if p_size>0:
            for _ in range(p_size):
                bag.append(empty_tile)
        return bag
            
            

    def __getitem__(self, idx: int) -> tuple:
        random.shuffle(self.img_dirs[idx])
        bag = []
        paths = self.img_dirs[idx]
        np.random.shuffle(paths)
        for i,img_path in enumerate(paths): #list of list of paths
            assert os.path.isfile(img_path), f"missing: {img_path}"
            tile = np.array(Image.open(img_path))[..., :3]
            black_bg = np.sum(tile, axis=2) == 0
            tile[black_bg, :] = 255
            mask_bg = np.mean(tile, axis=2) > self.white_thr
            bag.append(tile)
            if (i+1)==self.bag_size:
                break
        bag = self.pad_bag(bag)
        labels = self.to_one_hot(self.labels[idx])
        
        # augmentation
        if self.transforms:
            for i in range(len(bag)):
                bag[i] = self.transforms(Image.fromarray(bag[i]))
        #print(f"img dim: {img.shape}")
#         print(bag)
        return torch.stack(bag), torch.tensor(labels).to(int)

    def __len__(self) -> int:
        return len(self.data)

# ==============================
# ==============================

Let us define some standard image augmentaion procedures and color normalizations...

In [ ]:
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

TRAIN_TRANSFORM = T.Compose([
    T.CenterCrop(INPUT_SIZE),
    #T.RandomResizedCrop(512, interpolation=InterpolationMode.BICUBIC, antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ToTensor(),
#     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.Normalize(img_color_mean, img_color_std),  # custom
])

VALID_TRANSFORM = T.Compose([
    T.CenterCrop(INPUT_SIZE),
    T.ToTensor(),
#     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.Normalize(img_color_mean, img_color_std),  # custom
])

In [ ]:
dataset = CancerTilesDataset(df_train, DATASET_IMAGES, bag_size = 6, transforms = TRAIN_TRANSFORM)
bag, label = dataset[20]
print(bag.shape)
for item in bag:
    item = item.permute(1,2,0).detach().cpu().numpy()
    item = item*img_color_std +img_color_mean
    plt.imshow(item)
    plt.show()

The DataModule include creating training and validation dataset with given split and feading it to particular data loaders...

In [ ]:
import multiprocessing as mproc
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class CancerSubtypeDM(pl.LightningDataModule):

    def __init__(
        self,
        df_data,
        path_img_dir,
        batch_size,
        bag_size,
        num_workers: int = None,
        train_transforms = TRAIN_TRANSFORM,
        valid_transforms = VALID_TRANSFORM
    ):
        super().__init__()
        self.df_data = df_data
        self.path_img_dir = path_img_dir
        self.batch_size = batch_size
        self.bag_size = bag_size
        self.num_workers = num_workers or mproc.cpu_count()
        self.train_dataset = None
        self.valid_dataset = None
        self.train_transforms = train_transforms
        self.valid_transforms = valid_transforms

    def prepare_data(self):
        pass

    @property
    def num_classes(self) -> int:
        assert self.train_dataset and self.valid_dataset
        return len(set(self.train_dataset.labels_unique + self.valid_dataset.labels_unique))

    def setup(self, stage=None):
        self.train_dataset = CancerTilesDataset(
            self.df_data, self.path_img_dir, bag_size = self.bag_size, mode='train', transforms=self.train_transforms)
        print(f"training dataset: {len(self.train_dataset)}")
        self.valid_dataset = CancerTilesDataset(
            self.df_data, self.path_img_dir, bag_size = self.bag_size, mode='valid', transforms=self.valid_transforms,
            # as validation is subsampled it may happen that some labels are missing
            # and so created one-hot-encoding vector will be sorter
            labels_lut=self.train_dataset.labels_lut)
        print(f"validation dataset: {len(self.valid_dataset)}")

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.valid_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

    def test_dataloader(self):
        pass

# ==============================
# ==============================

dm = CancerSubtypeDM(
    df_train,
    DATASET_IMAGES,
    batch_size=BATCH_SIZE,
    bag_size = BAG_SIZE,
)
dm.setup()
print(dm.num_classes)

# quick view
fig = plt.figure(figsize=(3, 9))
for imgs, lbs in dm.train_dataloader():
    imgs = imgs[0,:9,:,:]
    lbs = lbs[0]
    print(f'batch labels: {torch.sum(lbs, axis=0)}')
    print(f'image size: {imgs[0].shape}')
    for i in range(3):
        ax = fig.add_subplot(3, 1, i + 1)
        #print(np.rollaxis(imgs[i].numpy(), 0, 3).shape)
        ax.imshow(imgs[i].permute(1,2,0).numpy())
        ax.set_title(lbs)
    break

## CNN Model

We start with some standard CNN models taken from TIMM.
Then we define Ligthning module including training and validation step and configure optimizer/schedular.

- **Schedulers's example**: https://www.kaggle.com/code/isbhargav/guide-to-pytorch-learning-rate-scheduling
- **Schedulers explained**: https://towardsdatascience.com/a-visual-guide-to-learning-rate-schedulers-in-pytorch-24bbb262c863#5407
- **TIMM models**: https://github.com/huggingface/pytorch-image-models/blob/main/results/results-imagenet.csv

In [ ]:
!pip install -q lion-pytorch adan-pytorch torch_optimizer

In [ ]:
import timm
import torch
import torchvision
from adan_pytorch import Adan
from lion_pytorch import Lion
from torch_optimizer import AdaBound, RAdam, Yogi

from torch import nn
from torch.nn import functional as F
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score

class AdaptiveConcatPool2d(torch.nn.Module):
    "Layer that concats `AdaptiveAvgPool2d` and `AdaptiveMaxPool2d`"
    def __init__(self, size=None):
        super().__init__()
        self.size = size or 1
        self.ap = torch.nn.AdaptiveAvgPool2d(self.size)
        self.mp = torch.nn.AdaptiveMaxPool2d(self.size)
    def forward(self, x): return torch.cat([self.mp(x), self.ap(x)], 1)


class LitCancerSubtype(pl.LightningModule):

    def __init__(self, enc,feature_dim, num_class = 5, lr: float = 1e-4, arch='classifier'):
        super().__init__()
#         self.conv_bag = torch.nn.Conv2d(in_channels=12*3,out_channels=3,kernel_size=3,padding=1)
#         self.net = net
#         self.arch = net.pretrained_cfg.get('architecture')
        self.num_classes = num_class
        self.train_accuracy = MulticlassAccuracy(num_classes=self.num_classes)
        self.val_accuracy = MulticlassAccuracy(num_classes=self.num_classes)
        self.val_f1_score = MulticlassF1Score(num_classes=self.num_classes)
        self.learn_rate = lr
        self.arch = arch
        self.enc = enc
        self.head = nn.Sequential(
            AdaptiveConcatPool2d(),
            torch.nn.Flatten(),
            nn.Linear(2*feature_dim,512),
            torch.nn.ReLU(),
            torch.nn.Mish(),
            torch.nn.LayerNorm(512),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(512,self.num_classes))

    def forward(self, x):
#         print(x.shape)
        batch,bag,c,h,w = x.shape
        x = x.view(batch*bag, c,h,w)
        x = self.enc(x)
#         print(x.shape)
        #x: bs*N x C x 4 x 4
        _,c_out,h_out,w_out = x.shape
        #concatenate the output for tiles into a single map
        x = x.view(batch,bag,c_out,h_out,w_out).permute(0,2,1,3,4).contiguous()
#         print(x.shape)
        x = x.view(batch,c_out,h_out*bag,w_out)
#         print(x.shape)
        x = self.head(x)
        
        #x: bs x n
        return x
    
    def compute_loss(self, y_hat, y):
        y_hat = torch.sigmoid(y_hat)
        return F.cross_entropy(y_hat, y.to(y_hat.dtype))

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        lbs = torch.argmax(y, axis=1)
        #print(f"{lbs=} ?= {y_hat=}")
        loss = self.compute_loss(y_hat, y)
        #print(f"{y=} ?= {y_hat=} -> {loss=}")
        self.log("train_loss", loss, logger=True, prog_bar=True)
        #print(f"{lb=} ?= {y_hat=} -> {self.train_accuracy(y_hat, lbs)}")
        self.log("train_acc", self.train_accuracy(y_hat, lbs), logger=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        lbs = torch.argmax(y, axis=1)
        loss = self.compute_loss(y_hat, y)
        self.log("valid_loss", loss, logger=True, prog_bar=False)
        self.log("valid_acc", self.val_accuracy(y_hat, lbs), logger=True, prog_bar=False)
        self.log("valid_f1", self.val_f1_score(y_hat, lbs), logger=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = AdaBound(self.parameters(), lr=self.learn_rate)
        #optimizer = RAdam(self.parameters(), lr=self.learn_rate)
        #optimizer = torch.optim.AdamW(self.parameters(), lr=self.learn_rate)
        #optimizer = Lion(self.parameters(), lr=self.learn_rate, weight_decay=1e-2)
        #optimizer = Adan(self.parameters(), lr=self.learn_rate * 10, betas=(0.02, 0.08, 0.01), weight_decay=0.02)
        
        #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        #    optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6, verbose=True)
        scheduler = torch.optim.lr_scheduler.CyclicLR(
          optimizer, base_lr=self.learn_rate, max_lr=self.learn_rate * 10,
          step_size_up=10, cycle_momentum=False, mode="triangular2", verbose=True)
        #scheduler = torch.optim.lr_scheduler.OneCycleLR(
        #    optimizer, max_lr=self.learn_rate * 5, steps_per_epoch=1, epochs=self.trainer.max_epochs)
        return [optimizer], [scheduler]

# ==============================
# ==============================

# see: https://pytorch.org/vision/stable/models.html
# net = timm.create_model('maxvit_tiny_tf_512', pretrained=True, num_classes=dm.num_classes)
arch = 'maxvit_tiny_tf_512'
net = timm.create_model('maxvit_tiny_tf_512', pretrained=True, num_classes=5)
enc = nn.Sequential(*list(net.children())[:-1])
model = LitCancerSubtype(enc=enc, feature_dim=512, lr=LR, arch='maxvit_tiny_tf_512')

## Training

We use Pytorch Lightning which allow us to drop all the boilet plate code and simplify all training just to use/call Trainer...

In [ ]:
logger = pl.loggers.CSVLogger(save_dir='logs/', name=model.arch)

# ==============================

trainer = pl.Trainer(
#     accelerator="cuda",
#     devices=2,
    # fast_dev_run=True,
    # callbacks=[swa],
    logger=logger,
    max_epochs=EPOCHS,
    precision='16-mixed',
    accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
    log_every_n_steps = 1,
    #val_check_interval=0.5,
)

# ==============================

# trainer.tune(model, datamodule=dm)
trainer.fit(model=model, datamodule=dm)

Quick visualization of the training process...

In [ ]:
import seaborn as sn

metrics = pd.read_csv(f'{trainer.logger.log_dir}/metrics.csv')
del metrics["step"]
metrics.set_index("epoch", inplace=True)
# display(metrics.dropna(axis=1, how="all").head())
g = sn.relplot(data=metrics, kind="line")
plt.gcf().set_size_inches(12, 4)
# plt.gca().set_yscale('log')
plt.grid()

Save the model!

In [ ]:
trainer.save_checkpoint("image_classification_model.pt")

Inference coming in https://www.kaggle.com/code/jirkaborovec/cancer-subtype-lightning-torch-inference-tiles